In [19]:
# ── Standard library ──────────────────────────────────────────────
import os
import sys
import json
import asyncio
from typing import TypedDict, Optional

# ── Path setup ────────────────────────────────────────────────────
sys.path.insert(0, os.path.abspath("../src"))

# ── Third party ───────────────────────────────────────────────────
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from loguru import logger

# ── Project modules ───────────────────────────────────────────────
from config import (
    OPENAI_API_KEY,
    AGENT_MODEL,
    FINAL_MODEL,
    CRITIQUE_MODEL,
    CHART_MODEL,
    MCP_SERVER_PATH,
    LANGFUSE_PUBLIC_KEY,
    LANGFUSE_SECRET_KEY,
    LANGFUSE_HOST
)
from mcp_server.client import MCPClient
from agent.tools import TOOLS
from agent.prompts import SYSTEM_PROMPT
from models.schemas import AgentAnswer, EvaluationResult
from observability.langfuse_setup import langfuse

# ── Load env ──────────────────────────────────────────────────────
load_dotenv()

# ── Clients ───────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Verify ────────────────────────────────────────────────────────
print(f"AGENT_MODEL:    {AGENT_MODEL}")
print(f"FINAL_MODEL:    {FINAL_MODEL}")
print(f"CRITIQUE_MODEL: {CRITIQUE_MODEL}")
print(f"CHART_MODEL:    {CHART_MODEL}")
print(f"MCP_SERVER:     {MCP_SERVER_PATH}")
print(f"langfuse:       {langfuse.auth_check()}")

AGENT_MODEL:    gpt-4.1-mini
FINAL_MODEL:    gpt-5-mini
CRITIQUE_MODEL: gpt-4.1-mini
CHART_MODEL:    gpt-5-mini
MCP_SERVER:     /Users/anshulgautam/Projects/datavis_agent/src/mcp_server/server.py
langfuse:       True


In [20]:
from typing import TypedDict

class AgentState(TypedDict):
    thread_id:         str
    question:          str
    messages:          list
    sql_result:        str
    df_columns:        list[str]
    sql_used:          str
    answer:            str
    chart_type:        str
    plotly_code:       str
    critique:          str
    failed_component:  str
    retry_count:       int
    is_approved:       bool
    report_sections:   list[dict]


def make_initial_state(question: str, thread_id: str = "default") -> AgentState:
    return AgentState(
        thread_id        = thread_id,
        question         = question,
        messages         = [],
        sql_result       = "",
        df_columns       = [],
        sql_used         = "",
        answer           = "",
        chart_type       = "none",
        plotly_code      = "",
        critique         = "",
        failed_component = "none",
        retry_count      = 0,
        is_approved      = False,
        report_sections  = []
    )


# ── DataFrame cache — local only, never sent to LLM ──────────────
_df_cache: dict = {}

def store_df(thread_id: str, df: pd.DataFrame):
    """Store df locally after SQL execution."""
    _df_cache[thread_id] = df
    logger.debug(f"[Cache] df stored for thread: {thread_id}")

def get_df(thread_id: str) -> pd.DataFrame | None:
    """Retrieve df for chart rendering — local only."""
    return _df_cache.get(thread_id)

In [21]:
# ── Test ──────────────────────────────────────────────────────────
state = make_initial_state("what are top 5 products by sales?", "test-1")
print(f"fields:    {list(state.keys())}")
print(f"thread_id: {state['thread_id']}")
print(f"question:  {state['question']}")

fields:    ['thread_id', 'question', 'messages', 'sql_result', 'df_columns', 'sql_used', 'answer', 'chart_type', 'plotly_code', 'critique', 'failed_component', 'retry_count', 'is_approved', 'report_sections']
thread_id: test-1
question:  what are top 5 products by sales?


In [39]:
# ── Install ───────────────────────────────────────────────────────
# pip install redis
import redis
import hashlib
import json
from typing import Optional

class QueryCache:
    """
    Redis cache for query results.
    Key   = SHA256 hash of question
    Value = serialised AgentState output
    TTL   = 1 hour by default
    """

    def __init__(self, host: str = "localhost", port: int = 6379, ttl: int = 3600):
        self.ttl = ttl
        try:
            self.client = redis.Redis(
                host            = host,
                port            = port,
                decode_responses = True
            )
            self.client.ping()
            self.enabled = True
            logger.info(f"Redis connected: {host}:{port}")
        except Exception as e:
            self.enabled = False
            logger.warning(f"Redis unavailable — cache disabled: {e}")

    def _make_key(self, question: str) -> str:
        """Deterministic key from question."""
        return f"query:{hashlib.sha256(question.strip().lower().encode()).hexdigest()}"

    def get(self, question: str) -> Optional[dict]:
        """Return cached result or None."""
        if not self.enabled:
            return None
        try:
            key  = self._make_key(question)
            data = self.client.get(key)
            if data:
                logger.info(f"[Cache] HIT: {question[:50]}")
                return json.loads(data)
            logger.info(f"[Cache] MISS: {question[:50]}")
            return None
        except Exception as e:
            logger.warning(f"[Cache] get error: {e}")
            return None

    def set(self, question: str, result: dict) -> bool:
        """Cache a result. Strips non-serialisable fields."""
        if not self.enabled:
            return False
        try:
            key = self._make_key(question)
            # only cache serialisable fields
            cacheable = {
                "answer":      result.get("answer", ""),
                "sql_used":    result.get("sql_used", ""),
                "chart_type":  result.get("chart_type", "none"),
                "plotly_code": result.get("plotly_code", ""),
                "df_json":     result.get("df_json", ""),
                "df_columns":  result.get("df_columns", [])
            }
            self.client.setex(key, self.ttl, json.dumps(cacheable))
            logger.info(f"[Cache] SET: {question[:50]} (TTL={self.ttl}s)")
            return True
        except Exception as e:
            logger.warning(f"[Cache] set error: {e}")
            return False

    def invalidate(self, question: str) -> bool:
        """Remove a cached result."""
        if not self.enabled:
            return False
        try:
            key = self._make_key(question)
            self.client.delete(key)
            logger.info(f"[Cache] INVALIDATED: {question[:50]}")
            return True
        except Exception as e:
            logger.warning(f"[Cache] invalidate error: {e}")
            return False

    def flush(self) -> bool:
        """Clear all cached queries."""
        if not self.enabled:
            return False
        try:
            self.client.flushdb()
            logger.info("[Cache] flushed")
            return True
        except Exception as e:
            logger.warning(f"[Cache] flush error: {e}")
            return False

    def stats(self) -> dict:
        """Return cache statistics."""
        if not self.enabled:
            return {"enabled": False}
        try:
            info = self.client.info()
            keys = self.client.dbsize()
            return {
                "enabled":     True,
                "keys":        keys,
                "memory_used": info.get("used_memory_human", "unknown"),
                "hits":        info.get("keyspace_hits", 0),
                "misses":      info.get("keyspace_misses", 0)
            }
        except Exception as e:
            return {"enabled": False, "error": str(e)}


# ── Initialise ────────────────────────────────────────────────────
cache = QueryCache(host="localhost", port=6379, ttl=3600)
print(cache.stats())

09:10:41 | INFO     | Redis connected: localhost:6379
{'enabled': True, 'keys': 0, 'memory_used': '1.02M', 'hits': 0, 'misses': 0}


In [23]:
async def planner_node(state: AgentState) -> AgentState:
    """
    Reads question.
    Creates brief plan for SQL agent.
    Returns updated state with messages set.
    """
    logger.info(f"[Planner] question: {state['question']}")

    response = client.chat.completions.create(
        model    = AGENT_MODEL,
        messages = [
            {
                "role": "system",
                "content": """You are a data analyst planner.
Given a question create a brief plan for what SQL queries are needed.
2-3 sentences max.
Do NOT write SQL — just describe what data is needed."""
            },
            {
                "role": "user",
                "content": f"Question: {state['question']}"
            }
        ]
    )

    plan = response.choices[0].message.content
    logger.info(f"[Planner] plan: {plan[:100]}...")

    return {
        **state,
        "messages": [
            {"role": "user", "content": f"Question: {state['question']}\nPlan: {plan}"}
        ]
    }

In [24]:
# ── Test ──────────────────────────────────────────────────────────
state = make_initial_state("what are top 5 products by sales?", "test-1")
state = await planner_node(state)

print(f"messages count: {len(state['messages'])}")
print(f"plan:\n{state['messages'][0]['content']}")

09:00:25 | INFO     | [Planner] question: what are top 5 products by sales?
09:00:27 | INFO     | [Planner] plan: You need to query the sales data grouped by product, summing the total sales amount for each product...
messages count: 1
plan:
Question: what are top 5 products by sales?
Plan: You need to query the sales data grouped by product, summing the total sales amount for each product. Then, sort these totals in descending order and select the top 5 products with the highest sales figures.


In [25]:
async def sql_agent_node(state: AgentState) -> AgentState:
    logger.info("[SQL Agent] running...")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": state["question"]}
    ]

    if state.get("critique") and state.get("retry_count", 0) > 0:
        messages.append({
            "role":    "user",
            "content": f"Previous attempt failed. Feedback: {state['critique']}. Please fix."
        })
        logger.info(f"[SQL Agent] retry {state['retry_count']} — feedback: {state['critique'][:80]}")

    last_df  = None
    sql_used = ""
    result   = ""

    async with MCPClient(MCP_SERVER_PATH) as mcp:
        while True:
            response = client.chat.completions.create(
                model    = AGENT_MODEL,
                messages = messages,
                tools    = TOOLS
            )

            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                break

            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)

                if tc.function.name == "list_tables":
                    logger.info("[SQL Agent] list_tables()")
                    result = await mcp.call_tool("list_tables")

                elif tc.function.name == "get_schema":
                    logger.info(f"[SQL Agent] get_schema({args.get('table_name')})")
                    result = await mcp.call_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    sql    = args.get("sql")
                    logger.info(f"[SQL Agent] run_sql: {sql}")
                    result = await mcp.call_tool("run_sql", {"sql": sql})

                    if not result.startswith("SQL_ERROR"):
                        try:
                            last_df  = pd.read_csv(
                                pd.io.common.StringIO(result),
                                sep    = r"\s{2,}",
                                engine = "python"
                            )
                            sql_used = sql
                            logger.info(f"[SQL Agent] rows: {len(last_df)}")
                        except Exception as e:
                            logger.warning(f"[SQL Agent] df parse error: {e}")
                    else:
                        logger.error(f"[SQL Agent] failed: {result}")

                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "content":      result
                })

        # ── Store df locally — never in state ────────────────────────
        if last_df is not None:
            store_df(state["thread_id"], last_df)

        # ── filter to plain dicts only before storing in state ───────────
        safe_messages = [
            m for m in messages
            if isinstance(m, dict)
        ]
    
        return {
            **state,
            "messages":   safe_messages,   # ← only dicts, no OpenAI objects
        "sql_result": result if last_df is not None else "",
        "df_columns": [str(c) for c in last_df.columns] if last_df is not None else [],
        "sql_used":   sql_used
    }

In [26]:
# ── Test ──────────────────────────────────────────────────────────
state = make_initial_state("what are top 5 products by sales?", "test-1")
state = await planner_node(state)
state = await sql_agent_node(state)

print(f"sql_used:   {state['sql_used'][:100]}")
print(f"df_columns: {state['df_columns']}")
print(f"sql_result: {state['sql_result'][:100]}")

# df in cache — not in state
df = get_df("test-1")
print(f"df shape:   {df.shape}")
print(f"df_json in state: {'df_json' in state}")  # should be False

09:00:32 | INFO     | [Planner] question: what are top 5 products by sales?
09:00:35 | INFO     | [Planner] plan: To answer this, query the sales transactions table to aggregate total sales amount per product, grou...
09:00:35 | INFO     | [SQL Agent] running...
09:00:36 | INFO     | MCP client connected to server
09:00:37 | INFO     | [SQL Agent] run_sql: SELECT product_name, SUM(sales) AS total_sales FROM sales_data GROUP BY product_name ORDER BY total_sales DESC LIMIT 5;
09:00:37 | ERROR    | [SQL Agent] failed: SQL_ERROR: Catalog Error: Table with name sales_data does not exist!
Did you mean "sqlite_master"?

LINE 1: SELECT product_name, SUM(sales) AS total_sales FROM sales_data GROUP BY product_name ORDER BY total_sales DESC...
                                                            ^
09:00:38 | INFO     | [SQL Agent] list_tables()
09:00:41 | INFO     | [SQL Agent] get_schema(Product)
09:00:41 | INFO     | [SQL Agent] get_schema(PurchaseOrderDetail)
09:00:42 | INFO     | [SQL 

In [27]:
async def chart_agent_node(state: AgentState) -> AgentState:
    """
    Generates chart type and plotly code.
    Uses df_columns + sql_result sample only — no real data sent to LLM.
    """
    logger.info("[Chart Agent] generating chart...")

    if not state["df_columns"] or not state["sql_result"]:
        logger.info("[Chart Agent] no data — skipping")
        return {**state, "chart_type": "none", "plotly_code": ""}

    response = client.chat.completions.create(
        model    = CHART_MODEL,
        messages = [
            {
                "role": "system",
                "content": """You are a data visualisation expert.
Return JSON only — no markdown:
{
  "chart_type": "bar|line|pie|scatter|histogram|table|none",
  "plotly_code": "import plotly.express as px\nfig = px.bar(...)"
}

Rules:
- df is already loaded as pandas DataFrame
- create figure called fig
- do NOT call fig.show()
- use double quotes only — never single quotes
- no escaped quotes
- no template parameter
- use ONLY the exact column names provided
- for single value results use chart_type none"""
            },
            {
                "role": "user",
                "content": f"""Question: {state['question']}
DataFrame columns: {state['df_columns']}
Data sample:
{state['sql_result'][:500]}"""
            }
        ]
    )

    try:
        raw     = response.choices[0].message.content
        cleaned = raw.replace("```json", "").replace("```", "").strip()
        parsed  = json.loads(cleaned)
        chart_type  = parsed.get("chart_type", "none")
        plotly_code = parsed.get("plotly_code", "")
        logger.info(f"[Chart Agent] chart_type: {chart_type}")
    except Exception as e:
        logger.warning(f"[Chart Agent] parse error: {e}")
        chart_type  = "none"
        plotly_code = ""

    return {**state, "chart_type": chart_type, "plotly_code": plotly_code}

In [28]:
# ── Test ──────────────────────────────────────────────────────────
state = await chart_agent_node(state)

print(f"chart_type:  {state['chart_type']}")
print(f"plotly_code: {state['plotly_code'][:150]}")

09:00:57 | INFO     | [Chart Agent] generating chart...
09:01:10 | INFO     | [Chart Agent] chart_type: bar
chart_type:  bar
plotly_code: import plotly.express as px
top5 = df.nlargest(5, "total_sales")
fig = px.bar(top5, x="product_name", y="total_sales", title="Top 5 Products by Sales"


In [29]:
async def answer_node(state: AgentState) -> AgentState:
    logger.info("[Answer] generating final answer...")

    df_context = (
        f"DataFrame columns: {state['df_columns']}. Use only these in plotly_code."
        if state["df_columns"]
        else "No data available. Set chart_type to none and plotly_code to empty string."
    )

    chart_instruction = f"Chart type already decided: {state['chart_type']}. Write plotly_code for this chart type only."

    messages = [
        {
            "role": "system",
            "content": """You are a data analyst.
Give a clear concise answer based on the data provided.
For plotly_code:
- df is already loaded as pandas DataFrame
- create figure called fig
- do NOT call fig.show()
- use double quotes only
- no escaped quotes
- no template parameter"""
        },
        {
            "role": "user",
            "content": f"""Question: {state['question']}
SQL used: {state['sql_used']}
Data: {state['sql_result'][:1000]}
{df_context}
{chart_instruction}
Give your final structured answer."""
        }
    ]

    final = client.beta.chat.completions.parse(
        model           = FINAL_MODEL,
        messages        = messages,
        response_format = AgentAnswer
    )

    result = final.choices[0].message.parsed
    logger.info(f"[Answer] answer: {result.answer[:100]}...")
    logger.info(f"[Answer] chart:  {state['chart_type']}")   # ← log state chart_type

    section = {
        "question":    state["question"],
        "answer":      result.answer,
        "sql_used":    result.sql_used,
        "chart_type":  state["chart_type"],  # ← chart_agent's decision
        "plotly_code": result.plotly_code,
        "df_columns":  state["df_columns"],
        "thread_id":   state["thread_id"]
    }

    return {
        **state,
        "answer":          result.answer,
        "sql_used":        result.sql_used,
        "chart_type":      state["chart_type"],   # ← chart_agent owns this
        "plotly_code":     result.plotly_code,
        "report_sections": state.get("report_sections", []) + [section]
    }

In [30]:
# ── Test ──────────────────────────────────────────────────────────
state = await answer_node(state)

print(f"answer:          {state['answer'][:150]}")
print(f"chart_type:      {state['chart_type']}")
print(f"report_sections: {len(state['report_sections'])} section")

# render chart locally — df never sent to LLM
df = get_df(state["thread_id"])
if state["plotly_code"] and df is not None:
    import plotly.express as px
    import plotly.graph_objects as go
    local_vars = {"df": df, "px": px, "go": go}
    exec(state["plotly_code"], local_vars)
    local_vars["fig"].show()

09:01:16 | INFO     | [Answer] generating final answer...
09:01:29 | INFO     | [Answer] answer: Top 5 products by total sales are:
1. HL Crankarm — 3,358,798.40
2. ML Mountain Pedal — 2,709,041.46...
09:01:29 | INFO     | [Answer] chart:  bar
answer:          Top 5 products by total sales are:
1. HL Crankarm — 3,358,798.40
2. ML Mountain Pedal — 2,709,041.46
3. ML Road Pedal — 2,390,330.70
4. Front Brakes —
chart_type:      bar
report_sections: 1 section


In [31]:
async def critique_node(state: AgentState) -> AgentState:
    """
    Evaluates complete output — SQL + chart + answer.
    Runs AFTER answer node.
    Routes retry to exact failed component.
    Max 2 retries before forcing approval.
    """
    logger.info("[Critique] evaluating...")

    # ── Max retries guard ─────────────────────────────────────────
    if state["retry_count"] >= 2:
        logger.warning("[Critique] max retries — forcing approval")
        return {**state, "is_approved": True, "failed_component": "none"}

    # ── Get row count from local cache ────────────────────────────
    df        = get_df(state["thread_id"])
    row_count = len(df) if df is not None else 0

    response = client.chat.completions.create(
        model    = CRITIQUE_MODEL,
        messages = [
            {
                "role": "system",
                "content": """You are a strict BI analyst critic.
Evaluate the complete output — SQL, chart and answer together.

Return JSON only:
{
  "approved": true/false,
  "reason": "specific actionable feedback",
  "failed_component": "sql|chart|answer|none"
}

failed_component rules:
- "sql"    → SQL wrong, missing, or no data returned
- "chart"  → chart type inappropriate for the data
- "answer" → answer vague, incomplete, or does not address question
- "none"   → everything correct, approved

Approve if ALL true:
- SQL correctly answers the question
- rows > 0 returned
- answer is meaningful and addresses question
- chart type is appropriate for the data

Reject if ANY true:
- SQL wrong or no data returned
- answer vague or off-topic
- chart inappropriate (e.g. bar for single number)"""
            },
            {
                "role": "user",
                "content": f"""Question: {state['question']}

SQL used: {state['sql_used']}
Rows returned: {row_count}
Data sample: {state['sql_result'][:400]}

Answer: {state['answer']}
Chart type: {state['chart_type']}"""
            }
        ]
    )

    try:
        raw              = response.choices[0].message.content
        cleaned          = raw.replace("```json", "").replace("```", "").strip()
        parsed           = json.loads(cleaned)
        approved         = parsed.get("approved", False)
        reason           = parsed.get("reason", "")
        failed_component = parsed.get("failed_component", "sql")
        logger.info(f"[Critique] approved: {approved} | failed: {failed_component} | reason: {reason[:80]}")
    except Exception as e:
        logger.warning(f"[Critique] parse error: {e} — approving by default")
        approved, reason, failed_component = True, "", "none"

    return {
        **state,
        "is_approved":      approved,
        "critique":         reason,
        "failed_component": failed_component,
        "retry_count":      state["retry_count"] + (0 if approved else 1)
    }

In [32]:
# ── Test ──────────────────────────────────────────────────────────
state = await critique_node(state)

print(f"approved:         {state['is_approved']}")
print(f"failed_component: {state['failed_component']}")
print(f"critique:         {state['critique'][:150]}")
print(f"retry_count:      {state['retry_count']}")

09:01:30 | INFO     | [Critique] evaluating...
09:01:32 | INFO     | [Critique] approved: True | failed: none | reason: The SQL correctly calculates the top 5 products by total sales, returning 5 rows
approved:         True
failed_component: none
critique:         The SQL correctly calculates the top 5 products by total sales, returning 5 rows with meaningful data. The answer clearly lists the top 5 products wit
retry_count:      0


In [33]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


def should_retry(state: AgentState) -> str:
    """
    Conditional edge after critique.
    Routes to exact failed component or END.
    """
    if state["is_approved"]:
        logger.info("[Graph] approved → END")
        return "end"

    failed = state.get("failed_component", "sql")
    logger.info(f"[Graph] rejected — failed: {failed} | retry: {state['retry_count']}")

    if failed == "chart":    return "chart_agent"
    elif failed == "answer": return "answer"
    else:                    return "sql_agent"


def build_single_query_graph():
    """Build and compile the single query graph."""

    graph = StateGraph(AgentState)

    # ── Nodes ─────────────────────────────────────────────────────
    graph.add_node("planner",     planner_node)
    graph.add_node("sql_agent",   sql_agent_node)
    graph.add_node("chart_agent", chart_agent_node)
    graph.add_node("answer",      answer_node)
    graph.add_node("critique",    critique_node)

    # ── Edges ─────────────────────────────────────────────────────
    graph.set_entry_point("planner")
    graph.add_edge("planner",     "sql_agent")
    graph.add_edge("sql_agent",   "chart_agent")
    graph.add_edge("chart_agent", "answer")
    graph.add_edge("answer",      "critique")

    # ── Conditional retry loop ────────────────────────────────────
    graph.add_conditional_edges(
        "critique",
        should_retry,
        {
            "end":         END,
            "sql_agent":   "sql_agent",
            "chart_agent": "chart_agent",
            "answer":      "answer"
        }
    )

    return graph.compile(checkpointer=MemorySaver())


# ── Compile once ──────────────────────────────────────────────────
single_query_graph = build_single_query_graph()
print("✅ graph compiled")
print(f"nodes: {list(single_query_graph.get_graph().nodes.keys())}")

✅ graph compiled
nodes: ['__start__', 'planner', 'sql_agent', 'chart_agent', 'answer', 'critique', '__end__']


In [34]:
import plotly.express as px
import plotly.graph_objects as go

async def run_query(question: str, thread_id: str = "test-1") -> dict:
    """
    Full pipeline:
    1. Check Redis cache
    2. Run graph if cache miss
    3. Store in Redis
    4. Render chart locally
    """

    # ── Step 1: Check cache ───────────────────────────────────────
    cached = cache.get(question)
    if cached:
        logger.info(f"[Pipeline] cache hit — returning instantly")
        return cached

    # ── Run graph ─────────────────────────────────────────────────
    logger.info(f"[Pipeline] cache miss — running graph")
    state  = make_initial_state(question, thread_id)
    config = {"configurable": {"thread_id": thread_id}}
    result = await single_query_graph.ainvoke(state, config=config)

    # ── Store in cache ────────────────────────────────────────────
    cache.set(question, result)

    # ── Render chart locally — only on fresh run ──────────────────
    df = get_df(thread_id)
    if result.get("plotly_code") and result.get("chart_type") != "none" and df is not None:
        try:
            local_vars = {"df": df, "px": px, "go": go}
            exec(result["plotly_code"], local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart render error: {e}")

    return result

In [35]:
# ── Test 1: First run — cache hit ────────────────────────────────
import time

start  = time.time()
result = await run_query("what are top 5 products by sales?", "test-1")
elapsed = round(time.time() - start, 2)

print(f"duration: {elapsed}s")   # should be < 0.1s
print(f"answer:   {result['answer'][:200]}")

09:02:39 | INFO     | [Pipeline] cache miss — running graph
09:02:39 | INFO     | [Planner] question: what are top 5 products by sales?
09:02:41 | INFO     | [Planner] plan: To answer this question, we need to query the sales data aggregated by product. Specifically, we sho...
09:02:41 | INFO     | [SQL Agent] running...
09:02:42 | INFO     | MCP client connected to server
09:02:43 | INFO     | [SQL Agent] run_sql: SELECT product_name, SUM(sales_amount) AS total_sales FROM sales_data GROUP BY product_name ORDER BY total_sales DESC LIMIT 5;
09:02:43 | ERROR    | [SQL Agent] failed: SQL_ERROR: Catalog Error: Table with name sales_data does not exist!
Did you mean "sqlite_master"?

LINE 1: SELECT product_name, SUM(sales_amount) AS total_sales FROM sales_data GROUP BY product_name ORDER BY total_sales DESC...
                                                                   ^
09:02:44 | INFO     | [SQL Agent] list_tables()
09:02:47 | INFO     | [SQL Agent] get_schema(Product)
09:02:47 | I

duration: 46.72s
answer:   Top 5 products by total sales (highest to lowest): 1) HL Crankarm — $3,358,798.40; 2) ML Mountain Pedal — $2,709,041.46; 3) ML Road Pedal — $2,390,330.70; 4) Front Brakes — $2,277,949.00; 5) Rear Brak


In [36]:
# ── Test 2: Second run — cache hit ────────────────────────────────
import time

start  = time.time()
result = await run_query("what are top 5 products by sales?", "test-1")
elapsed = round(time.time() - start, 2)

print(f"duration: {elapsed}s")   # should be < 0.1s
print(f"answer:   {result['answer']}")

09:03:42 | INFO     | [Pipeline] cache miss — running graph
09:03:42 | INFO     | [Planner] question: what are top 5 products by sales?
09:03:44 | INFO     | [Planner] plan: To answer this, we need to query the sales data table, aggregating total sales amounts for each prod...
09:03:44 | INFO     | [SQL Agent] running...
09:03:45 | INFO     | MCP client connected to server
09:03:47 | INFO     | [SQL Agent] run_sql: SELECT sales_order_line.product_id, SUM(sales_order_line.line_total) AS total_sales FROM sales_order_line GROUP BY sales_order_line.product_id ORDER BY total_sales DESC LIMIT 5;
09:03:47 | ERROR    | [SQL Agent] failed: SQL_ERROR: Catalog Error: Table with name sales_order_line does not exist!
Did you mean "sqlite_temp_master"?

LINE 1: ...t_id, SUM(sales_order_line.line_total) AS total_sales FROM sales_order_line GROUP BY sales_order_line.product_id ORDER...
                                                                      ^
09:03:48 | INFO     | [SQL Agent] list_tables

CancelledError: 

In [ ]:
# ── Test 3: Different question — cache miss ───────────────────────
result = await run_query("how many work order are there?", "test-2")

print(f"answer:     {result['answer'][:200]}")
print(f"chart_type: {result['chart_type']}")
print(f"cache stats: {cache.stats()}")